### 1. Persiapan dan Import Pustaka
Tahap ini dilakukan untuk mengimpor semua pustaka standar yang dibutuhkan untuk manipulasi data tabular (`pandas`), operasi array (`numpy`), pemrosesan gambar (`cv2` / OpenCV), dan manajemen file (`os`, `shutil`). `tqdm` digunakan untuk menampilkan *progress bar* agar kita tahu estimasi waktu prosesnya.

In [ ]:
import os
import pandas as pd
import numpy as np
import cv2
import shutil
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()

### 2. Konfigurasi Path, Label, dan Mode Balancing
Kontrol utama *pipeline*, variabel `BALANCING_MODE` dapat diganti dengan ketentuan:
- **`"undersample"`**: Membuang sebagian data mayoritas agar seimbang dengan data minoritas yang diaugmentasi. Cocok jika ukuran dataset ingin dijaga tetap kecil.
- **`"keep_all"`**: Mempertahankan seluruh data asli (tidak ada yang dibuang) dan hanya memperbanyak data minoritas melalui augmentasi. Sangat bagus agar model tidak kehilangan informasi dari data mayoritas.

In [ ]:
# Path Utama
BASE_DIR = os.getenv("BASE_DIR")
OUTPUT_DIR = os.getenv("OUTPUT_DIR")

# Path Input CSV & Folder Gambar
TRAIN_CSV = os.path.join(BASE_DIR, "Cleaned_TrainLabels.csv")
VAL_CSV = os.path.join(BASE_DIR, "Cleaned_ValidationLabels.csv")
TEST_CSV = os.path.join(BASE_DIR, "Cleaned_TestLabels.csv")

TRAIN_DIR = os.path.join(BASE_DIR, "Train")
VAL_DIR = os.path.join(BASE_DIR, "Validation")
TEST_DIR = os.path.join(BASE_DIR, "Test")

# Konfigurasi Label
IMAGE_COL = 'Image_Name' 
LABELS = ['Boredom', 'Engagement', 'Confusion', 'Frustration']

# --- KONFIGURASI BALANCING ---
BALANCING_MODE = "undersample" # Pilih: "undersample" atau "keep_all"

RARE_THRESHOLD = 300 
MAX_COMMON_IMAGES = 500 # Hanya terpakai jika mode "undersample"
DUPLICATION_FACTOR = 2  # Jumlah salinan augmentasi per gambar langka

### 3. Fungsi Augmentasi Gambar
Fungsi ini bertugas memberikan variasi pada data minoritas agar model (VLM/CNN) tidak sekadar menghafal gambar yang diulang-ulang, dengan menggunakan:
1. **Random Horizontal Flip** (Probabilitas 50%)
2. **Random Brightness** (Mengubah nilai *Value* pada ruang warna HSV secara acak antara -30 hingga +30)
3. **Slight Rotation** (Rotasi Ringan): Sudut -5 hingga +5 derajat untuk mensimulasikan webcam atau posisi duduk yang agak miring.
4. **Gaussian Blur**: Mensimulasikan lensa webcam yang kurang fokus (out of focus).
5. **Gaussian Noise**: Mensimulasikan bintik-bintik (grain) pada kamera laptop yang murah atau saat kondisi ruangan kurang cahaya.

In [ ]:
def augment_image(image):
    # 1. Random Horizontal Flip (50% probabilitas)
    if np.random.rand() > 0.5:
        image = cv2.flip(image, 1)
        
    # 2. Random Brightness (100% selalu diaplikasikan dengan nilai acak)
    value = np.random.randint(-30, 31)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    
    v = v.astype(np.int16)
    v = np.clip(v + value, 0, 255)
    v = v.astype(np.uint8)
    
    final_hsv = cv2.merge((h, s, v))
    image = cv2.cvtColor(final_hsv, cv2.COLOR_HSV2BGR)

    # 3. Random Slight Rotation (30% probabilitas)
    # Putar sedikit (-5 sampai +5 derajat) simulasi posisi kamera miring
    if np.random.rand() > 0.7:
        angle = np.random.uniform(-5, 5)
        h, w = image.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        # BORDER_REPLICATE agar ujung gambar yang kosong terisi warna piksel terdekat
        image = cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_REPLICATE)

    # 4. Random Gaussian Blur (30% probabilitas)
    # Simulasi webcam yang kurang fokus
    if np.random.rand() > 0.7:
        ksize = np.random.choice([3, 5]) # Ukuran blur ringan
        image = cv2.GaussianBlur(image, (ksize, ksize), 0)

    # 5. Random Gaussian Noise (30% probabilitas)
    # Simulasi noise/bintik pada kondisi low-light
    if np.random.rand() > 0.7:
        row, col, ch = image.shape
        mean = 0
        var = 15 # Intensitas noise (jangan terlalu tinggi agar wajah tidak hilang)
        sigma = var ** 0.5
        gauss = np.random.normal(mean, sigma, (row, col, ch))
        gauss = gauss.reshape(row, col, ch)
        noisy = image.astype(np.float32) + gauss
        image = np.clip(noisy, 0, 255).astype(np.uint8)

    return image

### 4. Fungsi Multi-Label Balancing (Khusus Data Train)
Ini adalah inti dari proses penyeimbangan kelas. Karena dataset DAiSEE bersifat *multi-label* (1 gambar punya 4 nilai emosi), maka akan dicari gambar-gambar yang memiliki setidaknya satu label minoritas. 

Berdasarkan `BALANCING_MODE`, data pasaran mayoritas akan dipotong atau dibiarkan utuh. Sementara itu, data minoritas akan selalu digandakan menggunakan fungsi augmentasi. Skrip juga akan menyimpan 1 sampel *before-after* augmentasi sebagai bukti visual.

In [ ]:
def process_multi_label_train(csv_path, img_dir, out_csv_path, out_img_dir):
    print(f"\n[INFO] Memulai Balancing TRAIN dengan mode: {BALANCING_MODE.upper()}")
    df = pd.read_csv(csv_path)
    os.makedirs(out_img_dir, exist_ok=True)
    
    # Deteksi gambar VIP (minoritas) berdasarkan threshold
    rare_conditions = pd.Series(False, index=df.index)
    for label in LABELS:
        counts = df[label].value_counts()
        rare_classes = counts[counts < RARE_THRESHOLD].index
        is_rare_for_this_label = df[label].isin(rare_classes)
        rare_conditions = rare_conditions | is_rare_for_this_label

    df_vip = df[rare_conditions].copy()
    df_common = df[~rare_conditions].copy()
    
    print(f"Total Gambar VIP (Minoritas): {len(df_vip)}")
    print(f"Total Gambar Pasaran (Mayoritas): {len(df_common)}")

    new_rows = []
    
    # --- PROSES GAMBAR PASARAN (MAYORITAS) ---
    if BALANCING_MODE == "undersample" and len(df_common) > MAX_COMMON_IMAGES:
        print(f"[INFO] Memotong data mayoritas menjadi {MAX_COMMON_IMAGES} gambar...")
        df_common_sampled = df_common.sample(n=MAX_COMMON_IMAGES, random_state=42)
    else:
        print(f"[INFO] Mempertahankan SELURUH data mayoritas ({len(df_common)} gambar)...")
        df_common_sampled = df_common

    for _, row in tqdm(df_common_sampled.iterrows(), total=len(df_common_sampled), desc="Menyalin Gambar Mayoritas"):
        img_name = str(row[IMAGE_COL])
        if not (img_name.endswith('.jpg') or img_name.endswith('.png')): img_name += '.jpg'
        
        src_path = os.path.join(img_dir, img_name)
        dst_path = os.path.join(out_img_dir, img_name)
        
        if os.path.exists(src_path):
            shutil.copy2(src_path, dst_path)
            new_rows.append(row)

    # --- PROSES OVERSAMPLING & AUGMENTASI GAMBAR ---
    sample_saved = False 
    
    for _, row in tqdm(df_vip.iterrows(), total=len(df_vip), desc="Augmentasi Data Minoritas"):
        img_name = str(row[IMAGE_COL])
        name_part, ext_part = os.path.splitext(img_name)
        if not ext_part: ext_part = '.jpg'; img_name += ext_part
            
        src_path = os.path.join(img_dir, img_name)
        
        if os.path.exists(src_path):
            # 1. Salin gambar asli
            dst_path = os.path.join(out_img_dir, img_name)
            shutil.copy2(src_path, dst_path)
            new_rows.append(row)
            
            # 2. Augmentasi
            img = cv2.imread(src_path)
            if img is not None:
                for i in range(DUPLICATION_FACTOR):
                    img_aug = augment_image(img)
                    
                    # Simpan contoh before-after (hanya 1 kali)
                    if not sample_saved:
                        img_display = img.copy()
                        img_aug_display = img_aug.copy()
                        cv2.putText(img_display, "Before", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                        cv2.putText(img_aug_display, "After", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                        comparison = cv2.hconcat([img_display, img_aug_display])
                        cv2.imwrite(os.path.join(OUTPUT_DIR, "sample_before_after.jpg"), comparison)
                        sample_saved = True

                    new_img_name = f"{name_part}_aug_{i}{ext_part}"
                    dst_path_aug = os.path.join(out_img_dir, new_img_name)
                    cv2.imwrite(dst_path_aug, img_aug)
                    
                    new_row = row.copy()
                    new_row[IMAGE_COL] = new_img_name.replace(ext_part, '') 
                    new_rows.append(new_row)

    # Simpan CSV hasil akhir Train
    df_balanced = pd.DataFrame(new_rows)
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
    df_balanced.to_csv(out_csv_path, index=False)
    print(f"\n[SELESAI] Data Train tersimpan. Total gambar Train sekarang: {len(df_balanced)}")

### 5. Fungsi Copy Data Validation & Test
Sangat penting dalam *Machine Learning* untuk **tidak memanipulasi (balancing/augmentasi) data validasi dan testing**. Data evaluasi harus merepresentasikan kondisi asli dunia nyata. Oleh karena itu, fungsi ini hanya bertugas menyalin gambar dan file CSV ke direktori output yang baru tanpa mengubah komposisi kelasnya.

In [ ]:
def copy_dataset(csv_path, img_dir, out_csv_path, out_img_dir, split_name):
    print(f"\n[INFO] Menyalin Data {split_name}...")
    os.makedirs(out_img_dir, exist_ok=True)
    df = pd.read_csv(csv_path)
    new_rows = []
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Proses {split_name}"):
        img_name = str(row[IMAGE_COL])
        if not (img_name.endswith('.jpg') or img_name.endswith('.png')): img_name += '.jpg'
            
        src_path = os.path.join(img_dir, img_name)
        if os.path.exists(src_path):
            shutil.copy2(src_path, os.path.join(out_img_dir, img_name))
            new_rows.append(row)
            
    pd.DataFrame(new_rows).to_csv(out_csv_path, index=False)

### 6. Eksekusi Utama Pipeline
Sel ini mengikat seluruh fungsi yang sudah didefinisikan sebelumnya. Ia akan membuat direktori output jika belum ada, memproses data latih (Train) untuk di-*balance*, dan menyalin data uji (Validation & Test) dengan rapi.

In [ ]:
if __name__ == "__main__":
    OUT_TRAIN_DIR = os.path.join(OUTPUT_DIR, "Train")
    OUT_VAL_DIR = os.path.join(OUTPUT_DIR, "Validation")
    OUT_TEST_DIR = os.path.join(OUTPUT_DIR, "Test")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    process_multi_label_train(
        TRAIN_CSV, 
        TRAIN_DIR, 
        os.path.join(OUTPUT_DIR, "Balanced_TrainLabels.csv"), 
        OUT_TRAIN_DIR
    )
    
    copy_dataset(
        VAL_CSV, 
        VAL_DIR, 
        os.path.join(OUTPUT_DIR, "Balanced_ValidationLabels.csv"), 
        OUT_VAL_DIR, 
        "Validation"
    )
    
    copy_dataset(
        TEST_CSV, 
        TEST_DIR, 
        os.path.join(OUTPUT_DIR, "Balanced_TestLabels.csv"), 
        OUT_TEST_DIR, 
        "Test"
    )
    
    print("\n[BERHASIL] Seluruh proses pipeline telah selesai!")
    print(f"Cek folder: {OUTPUT_DIR} untuk melihat gambar 'sample_before_after.jpg'")

In [7]:
import mediapipe as mp
print(mp.__version__)

0.10.32


In [8]:
import mediapipe as mp
import os
import pandas as pd
import cv2
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from dotenv import load_dotenv


load_dotenv() 

BASE_DIR = os.getenv("OUTPUT_DIR")
load_dotenv() 
import urllib.request
urllib.request.urlretrieve(
    "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
    "face_landmarker.task"
)

base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    num_faces=1
)
face_mesh = vision.FaceLandmarker.create_from_options(options)
print("Face Landmarker berhasil diinisialisasi!")

LEFT_EYE   = [33, 160, 158, 133, 153, 144]
RIGHT_EYE  = [362, 385, 387, 263, 373, 380]
MOUTH      = [78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308, 324, 318, 402, 317, 14, 87, 178, 88, 95]
LEFT_BROW  = [46, 53, 52, 65, 55]
RIGHT_BROW = [276, 283, 282, 295, 285]
NOSE       = [1, 2, 98, 327, 168, 6]
JAW        = [152, 148, 176, 149, 150, 136, 172, 377, 400, 378, 379, 365, 397]

def get_bbox(landmarks, indices, img_w, img_h, padding=15):
    xs = [landmarks[i].x * img_w for i in indices]
    ys = [landmarks[i].y * img_h for i in indices]

    xmin = max(0,     int(min(xs)) - padding)
    ymin = max(0,     int(min(ys)) - padding)
    xmax = min(img_w, int(max(xs)) + padding)
    ymax = min(img_h, int(max(ys)) + padding)

    return f"<box>({ymin},{xmin}),({ymax},{xmax})</box>"

BASE_DIR = os.getenv("OUTPUT_DIR")
splits = [
    {
        "name": "Train",
        "csv_in":  os.path.join(BASE_DIR, "Balanced_TrainLabels.csv"),
        "img_dir": os.path.join(BASE_DIR, "Train"),
        "csv_out": "mediapipe_bboxes_train.csv"
    },
    {
        "name": "Validation",
        "csv_in":  os.path.join(BASE_DIR, "Balanced_ValidationLabels.csv"),
        "img_dir": os.path.join(BASE_DIR, "Validation"),
        "csv_out": "mediapipe_bboxes_val.csv"
    },
    {
        "name": "Test",
        "csv_in":  os.path.join(BASE_DIR, "Balanced_TestLabels.csv"),
        "img_dir": os.path.join(BASE_DIR, "Test"),
        "csv_out": "mediapipe_bboxes_test.csv"
    }
]

LABEL_COLS = ['Boredom', 'Engagement', 'Confusion', 'Frustration']

for split in splits:
    print(f"\n{'='*50}")
    print(f"Memulai ekstraksi: {split['name']}")
    print(f"{'='*50}")

    if not os.path.exists(split['csv_in']):
        print(f"[SKIP] File tidak ditemukan: {split['csv_in']}")
        continue

    df = pd.read_csv(split['csv_in'])
    total = len(df)
    extracted_data = []
    skipped = 0

    for index, row in df.iterrows():
        if index % 100 == 0:
            print(f"  Progress: {index}/{total} ({index/total*100:.1f}%)", end='\r')

        clip_id    = str(row['Image_Name']).split('.')[0]
        image_path = os.path.join(split['img_dir'], f"{clip_id}.jpg")

        if not os.path.exists(image_path):
            skipped += 1
            continue

        image = cv2.imread(image_path)
        if image is None:
            skipped += 1
            continue

        img_h, img_w = image.shape[:2]
        rgb_image    = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)


        results = face_mesh.detect(mp_image)


        if not results.face_landmarks:
            skipped += 1
            continue

        lm = results.face_landmarks[0] 

        labels = {col: row[col] if col in row.index else None for col in LABEL_COLS}

        extracted_data.append({
            'ClipID':        clip_id,
            'ImagePath':     image_path,
            **labels,
            'LeftEye_Box':   get_bbox(lm, LEFT_EYE,   img_w, img_h),
            'RightEye_Box':  get_bbox(lm, RIGHT_EYE,  img_w, img_h),
            'LeftBrow_Box':  get_bbox(lm, LEFT_BROW,  img_w, img_h),
            'RightBrow_Box': get_bbox(lm, RIGHT_BROW, img_w, img_h),
            'Nose_Box':      get_bbox(lm, NOSE,        img_w, img_h),
            'Mouth_Box':     get_bbox(lm, MOUTH,       img_w, img_h),
            'Jaw_Box':       get_bbox(lm, JAW,         img_w, img_h),
        })

    out_df = pd.DataFrame(extracted_data)
    out_df.to_csv(split['csv_out'], index=False)

    print(f"\n[DONE] {split['name']}:")
    print(f"  Berhasil : {len(extracted_data)} gambar")
    print(f"  Dilewati : {skipped} gambar")
    print(f"  Disimpan : {split['csv_out']}")

face_mesh.close()
print("\nSemua split selesai diproses!")

W0000 00:00:1774975987.654865   70769 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1774975987.657815   70769 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1774975987.658943   70792 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: Mesa Intel(R) Iris(R) Xe Graphics (RPL-P)
W0000 00:00:1774975987.661504   70773 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774975987.670960   70788 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Face Landmarker berhasil diinisialisasi!

Memulai ekstraksi: Train
  Progress: 1900/1982 (95.9%)
[DONE] Train:
  Berhasil : 1982 gambar
  Dilewati : 0 gambar
  Disimpan : mediapipe_bboxes_train.csv

Memulai ekstraksi: Validation
  Progress: 2800/2858 (98.0%)
[DONE] Validation:
  Berhasil : 2857 gambar
  Dilewati : 1 gambar
  Disimpan : mediapipe_bboxes_val.csv

Memulai ekstraksi: Test
  Progress: 3200/3275 (97.7%)
[DONE] Test:
  Berhasil : 3271 gambar
  Dilewati : 4 gambar
  Disimpan : mediapipe_bboxes_test.csv

Semua split selesai diproses!


In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import matplotlib.pyplot as plt

base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    num_faces=1
)
face_mesh = vision.FaceLandmarker.create_from_options(options)

LEFT_EYE_INDICES       = [33, 160, 158, 133, 153, 144]
RIGHT_EYE_INDICES      = [362, 385, 387, 263, 373, 380]
MOUTH_INDICES          = [78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308, 324, 318, 402, 317, 14, 87, 178, 88, 95]
LEFT_EYEBROW_INDICES   = [46, 53, 52, 65, 55]
RIGHT_EYEBROW_INDICES  = [276, 283, 282, 295, 285]
NOSE_INDICES           = [1, 2, 98, 327, 168, 6]
JAW_INDICES            = [152, 148, 176, 149, 150, 136, 172, 377, 400, 378, 379, 365, 397]

def get_pixel_bbox(landmarks, indices, img_width, img_height, padding=15):
    xs = [landmarks[i].x for i in indices]
    ys = [landmarks[i].y for i in indices]
    xmin = max(0,          int(min(xs) * img_width)  - padding)
    ymin = max(0,          int(min(ys) * img_height) - padding)
    xmax = min(img_width,  int(max(xs) * img_width)  + padding)
    ymax = min(img_height, int(max(ys) * img_height) + padding)
    return (xmin, ymin), (xmax, ymax)


sample_image_path = ''

image = cv2.imread(sample_image_path)

if image is None:
    print(f"Gambar tidak ditemukan: {sample_image_path}")
else:
    img_height, img_width, _ = image.shape
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
    results  = face_mesh.detect(mp_image)

    if results.face_landmarks:
        lm = results.face_landmarks[0] 

        boxes = {
            "Left Eye":   (get_pixel_bbox(lm, LEFT_EYE_INDICES,      img_width, img_height), (0, 255, 0)),
            "Right Eye":  (get_pixel_bbox(lm, RIGHT_EYE_INDICES,     img_width, img_height), (0, 255, 0)),
            "Left Brow":  (get_pixel_bbox(lm, LEFT_EYEBROW_INDICES,  img_width, img_height), (255, 0, 0)),
            "Right Brow": (get_pixel_bbox(lm, RIGHT_EYEBROW_INDICES, img_width, img_height), (255, 0, 0)),
            "Mouth":      (get_pixel_bbox(lm, MOUTH_INDICES,         img_width, img_height), (0, 0, 255)),
            "Nose":       (get_pixel_bbox(lm, NOSE_INDICES,          img_width, img_height), (255, 255, 0)),
            "Jaw":        (get_pixel_bbox(lm, JAW_INDICES,           img_width, img_height), (255, 0, 255)),
        }

        for name, (bbox, color) in boxes.items():
            pt1, pt2 = bbox
            cv2.rectangle(image_rgb, pt1, pt2, color, 2)
            cv2.putText(image_rgb, name, (pt1[0], pt1[1] - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

        plt.figure(figsize=(10, 10))
        plt.imshow(image_rgb)
        plt.axis('off')
        plt.title("Visualisasi ROI MediaPipe", fontsize=16)
        plt.show()

    else:
        print("Wajah tidak terdeteksi.")

face_mesh.close()

In [ ]:
import pandas as pd

df = pd.read_csv('')

print("Distribusi Engagement:")
print(df["Engagement"].value_counts().sort_index())